## Процессы

### Задание 1

Используя multiprocessing.Pool, реализуйте параллельную обработку списка чисел. Напишите функцию, которая возводит число в квадрат. Создайте пул из 4 процессов и используйте map.

In [1]:
from multiprocessing import Process, Value, Lock
import multiprocessing
import os
import time
import threading
import queue
import random

In [2]:
numbers = [1, 2, 3, 4, 5]

def square_number(x):
    return x ** 2

with multiprocessing.Pool(processes=4) as pool:
    squared_numbers = pool.map(square_number, numbers)

print(f"Исходные числа: {numbers}")
print(f"Квадраты: {squared_numbers}")

Исходные числа: [1, 2, 3, 4, 5]
Квадраты: [1, 4, 9, 16, 25]


### Задание 2

Создайте общую переменную Value и защитите её с помощью Lock, чтобы два процесса, одновременно прибавляя единицу 1000 раз, выдали в итоге ровно 2000.

In [3]:
def increment(shared_val, lock):
    for _ in range(1000):
        with lock:
            shared_val.value += 1

counter = Value('i', 0)
lock = Lock()

p1 = Process(target=increment, args=(counter, lock))
p2 = Process(target=increment, args=(counter, lock))

p1.start()
p2.start()

p1.join()
p2.join()

print(f"Итог: {counter.value}")

Итог: 2000


### Задание 3

Реализуйте передачу строки через Queue. Дочерний процесс должен получить строку и напечатать её вместе со своим PID (используйте os.getpid()).

In [4]:
import os

def worker(q):
    message = q.get()
    print(f"Дочерний процесс PID={os.getpid()} получил сообщение: {message}")

mp_queue = multiprocessing.Queue()
p = multiprocessing.Process(target=worker, args=(mp_queue,))
p.start()

mp_queue.put("Привет из родительского процесса!")

p.join()

Дочерний процесс PID=11589 получил сообщение: Привет из родительского процесса!


### Задание 4

Условие:

Создайте два процесса, которые «общаются» друг с другом.

- Процесс sender отправляет число в Pipe.

- Процесс receiver получает это число, возводит его в квадрат и отправляет результат обратно в этот же Pipe.

- Процесс sender получает ответ и выводит его на экран.

In [5]:
def worker_receiver(conn):
    number = conn.recv()
    result = number ** 2
    conn.send(result)
    conn.close()

def worker_sender(conn, number):
    conn.send(number)
    result = conn.recv()
    print(f"Sender отправил число {number}, получил ответ: {result}")
    conn.close()

sender_conn, receiver_conn = multiprocessing.Pipe()

sender = Process(target=worker_sender, args=(sender_conn, 7))
receiver = Process(target=worker_receiver, args=(receiver_conn,))

receiver.start()
sender.start()

sender.join()
receiver.join()

Sender отправил число 7, получил ответ: 49


### Задание 5

Используйте Manager, чтобы создать общий словарь. Каждый процесс должен записать в него свое имя (ключ) и свой PID (значение).

In [6]:
def register_process(shared_dict):
    process_name = multiprocessing.current_process().name
    pid = os.getpid()
    shared_dict[process_name] = (pid, "данные успешно записаны")

manager = multiprocessing.Manager()
shared_dict = manager.dict()

processes = []
for i in range(4):
    p = multiprocessing.Process(
        target=register_process,
        args=(shared_dict,),
        name=f"Worker-{i + 1}"
    )
    processes.append(p)
    p.start()

for p in processes:
    p.join()

for name, info in shared_dict.items():
    print(f"Процесс {name} (PID {info[0]}) записал: {info[1]}")

manager.shutdown()

Процесс Worker-2 (PID 11608) записал: данные успешно записаны
Процесс Worker-1 (PID 11607) записал: данные успешно записаны
Процесс Worker-4 (PID 11618) записал: данные успешно записаны
Процесс Worker-3 (PID 11610) записал: данные успешно записаны


##  Потоки

In [7]:
import random
import queue
import threading
import time

## Задание 1

Представьте, что этот код запущен в Python 3.13+ с флагом --disable-gil (free-threading)

Задача: Измените код внутри функции increment так, чтобы возникла ситуация Race Condition.

При запуске 10 потоков итоговое значение counter должно быть меньше 1 000 000 (c GIL будет 1 000 000).

Подсказка: сделайте операцию инкремента неатомарной, разбив её на чтение и запись.

In [8]:
counter = 0

def increment():
    global counter
    for _ in range(100000):
        current_value = counter
        new_value = current_value + 1
        counter = new_value

threads = [threading.Thread(target=increment) for _ in range(10)]
for t in threads:
    t.start()
for t in threads:
    t.join()

print(f"Итоговый счетчик: {counter}")

Итоговый счетчик: 1000000


### Задание 2

Допишите логику синхронизации в функции worker.
Все 5 потоков должны завершить "Фазу 1" и только потом одновременно начать "Фазу 2".

Использовать threading.Barrier запрещено.

Используйте Lock и Event.

In [9]:
def worker(worker_id, info):
    print(f"[Поток {worker_id}] Начинает Фазу 1...")
    time.sleep(random.uniform(0.5, 1.5))
    print(f"[Поток {worker_id}] Завершил Фазу 1")

    with info["lock"]:
        info["counter"] += 1
        if info["counter"] == 5:
            info["event"].set()

    info["event"].wait()

    print(f"[Поток {worker_id}] >>> ПЕРЕШЕЛ К ФАЗЕ 2")

shared_data = {
    "counter": 0,
    "lock": threading.Lock(),
    "event": threading.Event()
}

threads = []
for i in range(5):
    t = threading.Thread(target=worker, args=(i, shared_data))
    threads.append(t)
    t.start()

for t in threads:
    t.join()

print()
print("Все потоки успешно прошли барьер.")

[Поток 0] Начинает Фазу 1...[Поток 1] Начинает Фазу 1...
[Поток 2] Начинает Фазу 1...
[Поток 3] Начинает Фазу 1...

[Поток 4] Начинает Фазу 1...
[Поток 2] Завершил Фазу 1
[Поток 1] Завершил Фазу 1
[Поток 3] Завершил Фазу 1
[Поток 4] Завершил Фазу 1
[Поток 0] Завершил Фазу 1
[Поток 0] >>> ПЕРЕШЕЛ К ФАЗЕ 2
[Поток 1] >>> ПЕРЕШЕЛ К ФАЗЕ 2
[Поток 2] >>> ПЕРЕШЕЛ К ФАЗЕ 2
[Поток 3] >>> ПЕРЕШЕЛ К ФАЗЕ 2
[Поток 4] >>> ПЕРЕШЕЛ К ФАЗЕ 2

Все потоки успешно прошли барьер.


### Задание 3

Задача: Реализуйте функцию воркера, которая обрабатывает задачи из очереди.

Поток должен корректно завершиться (выйти из цикла) в двух ситуациях:
1. Если в очередь пришел специальный сигнал остановки — None (poison pill).
2. Если переданный Event (stop_event) перешел в состояние True.

ВАЖНО: Поток не должен блокироваться на q.get() вечно, если работа должна быть прекращена.

In [10]:
import queue
import threading
import time

def smart_worker(q, stop_event):
    while True:
        if stop_event.is_set():
            print("Worker остановлен через Event")
            break

        try:
            task = q.get(timeout=0.2)
        except queue.Empty:
            continue

        if task is None:
            print("Worker получил poison pill и завершает работу")
            q.task_done()
            break

        print(f"Worker обрабатывает задачу: {task}")
        time.sleep(0.5)
        q.task_done()


task_queue = queue.Queue()
stop_signal = threading.Event()

worker_thread = threading.Thread(target=smart_worker, args=(task_queue, stop_signal))
worker_thread.start()

task_queue.put("Обработать данные пользователя")
time.sleep(1)

print("Подаем сигнал остановки через Event...")
stop_signal.set()
worker_thread.join()

print("Программа успешно завершена.")

Worker обрабатывает задачу: Обработать данные пользователя
Подаем сигнал остановки через Event...
Worker остановлен через Event
Программа успешно завершена.


### Задание 4

Задача: Ниже представлен код, который гарантированно приводит к зависанию (Deadlock).

Поток 1 захватывает замок A и ждет B. Поток 2 захватывает замок B и ждет A.

Исправьте одну из функций так, чтобы дедлока не возникало.

Соблюдайте правило: все потоки должны захватывать одни и те же замки в одинаковом порядке.

In [11]:
lock_a = threading.Lock()
lock_b = threading.Lock()

def process_one():
    with lock_a:
        print("[P1] Захватил Lock A, думаю...")
        time.sleep(0.5)
        print("[P1] Пытаюсь захватить Lock B...")
        with lock_b:
            print("[P1] Успех! Выполнил задачу.")

def process_two():
    with lock_a:
        print("[P2] Захватил Lock A, думаю...")
        time.sleep(0.5)
        print("[P2] Пытаюсь захватить Lock B...")
        with lock_b:
            print("[P2] Успех! Выполнил задачу.")

t1 = threading.Thread(target=process_one)
t2 = threading.Thread(target=process_two)

t1.start()
t2.start()

t1.join()
t2.join()
print("Программа завершена без дедлока!")

[P1] Захватил Lock A, думаю...
[P1] Пытаюсь захватить Lock B...
[P1] Успех! Выполнил задачу.
[P2] Захватил Lock A, думаю...
[P2] Пытаюсь захватить Lock B...
[P2] Успех! Выполнил задачу.
Программа завершена без дедлока!


### Задание 5

Задача: Реализуйте паттерн Singleton (Одиночка) так, чтобы при вызове DatabaseConnection() из 100 разных потоков, все они получили ссылку на один и тот же объект в памяти.

ВАЖНО: Используйте механизм Double-Checked Locking для оптимизации производительности.

In [12]:
class DatabaseConnection:
    _instance = None
    _lock = threading.Lock()

    def __new__(cls):
        if cls._instance is None:
            with cls._lock:
                if cls._instance is None:
                    cls._instance = super().__new__(cls)
        return cls._instance


def test_singleton():
    obj = DatabaseConnection()
    print(f"Поток {threading.current_thread().name} получил объект ID: {id(obj)}")

threads = []
for i in range(10):
    t = threading.Thread(target=test_singleton, name=f"T-{i}")
    threads.append(t)
    t.start()

for t in threads:
    t.join()

Поток T-0 получил объект ID: 137828218405520
Поток T-1 получил объект ID: 137828218405520
Поток T-2 получил объект ID: 137828218405520
Поток T-3 получил объект ID: 137828218405520
Поток T-4 получил объект ID: 137828218405520
Поток T-5 получил объект ID: 137828218405520
Поток T-6 получил объект ID: 137828218405520
Поток T-7 получил объект ID: 137828218405520
Поток T-8 получил объект ID: 137828218405520
Поток T-9 получил объект ID: 137828218405520


## asyncio

In [13]:
import asyncio
import time
import nest_asyncio
nest_asyncio.apply()

### Задание 1

1. Напишите корутину fetch_data(id, delay), которая имитирует загрузку данных: выводит сообщение о старте, спит delay секунд (асинхронно!) и возвращает строку f"Data-{id}".
2. Напишите корутину main(), которая запускает 3 таких загрузки конкурентно с задержками 1, 2 и 3 секунды соответственно.
3. Выведите итоговый список результатов и общее время выполнения (оно должно быть около 3 сек, а не 6).

In [14]:
async def fetch_data(data_id, delay):
    print(f"Старт загрузки Data-{data_id}, задержка {delay} сек")
    await asyncio.sleep(delay)
    print(f"Загрузка Data-{data_id} завершена")
    return f"Data-{data_id}"

async def main():
    start_time = time.perf_counter()

    results = await asyncio.gather(
        fetch_data(1, 1),
        fetch_data(2, 2),
        fetch_data(3, 3)
    )

    end_time = time.perf_counter()
    print(f"Результаты: {results}")
    print(f"Затрачено времени: {end_time - start_time:.2f} сек")

asyncio.run(main())

Старт загрузки Data-1, задержка 1 сек
Старт загрузки Data-2, задержка 2 сек
Старт загрузки Data-3, задержка 3 сек
Загрузка Data-1 завершена
Загрузка Data-2 завершена
Загрузка Data-3 завершена
Результаты: ['Data-1', 'Data-2', 'Data-3']
Затрачено времени: 3.00 сек


### Задание 2

1. Напишите корутину slow_api_call(), которая спит 5 секунд и возвращает "Success".
2. В main() вызовите эту корутину, но ограничьте её выполнение 2 секундами с помощью asyncio.wait_for.
3. Обработайте исключение asyncio.TimeoutError, чтобы программа не падала, а выводила "API запрос занял слишком много времени!".

In [15]:
async def slow_api_call():
    await asyncio.sleep(5)
    return "Success"

async def main():
    try:
        result = await asyncio.wait_for(slow_api_call(), timeout=2)
        print(result)
    except asyncio.TimeoutError:
        print("API запрос занял слишком много времени!")

asyncio.run(main())

API запрос занял слишком много времени!


### Задание 3

Представьте, что вам нужно проверить доступность 20 сайтов.
Если запустить все 20 запросов одновременно, сервер может расценить это как атаку.

1. Напишите корутину fetch_url(url, semaphore), которая:
    - Использует семафор для ограничения входа (одновременно не более 3-х).
    - Имитирует запрос (asyncio.sleep от 1 до 3 сек).
    - Выводит сообщение: "[Запрос] Проверка {url} началась".
    - После "ответа" выводит: "[Готово] {url} проверен".
2. В main() создайте список из 20 условных URL (например, site_1, site_2...) и запустите их конкурентно, но с ограничением семафора.

In [16]:
import random

async def fetch_url(url, semaphore):
    async with semaphore:
        print(f"[Запрос] Проверка {url} началась")
        await asyncio.sleep(random.randint(1, 3))
        print(f"[Готово] {url} проверен")
        return url

async def main():
    sem = asyncio.Semaphore(3)
    urls = [f"https://site_{i}.com" for i in range(1, 21)]

    tasks = [asyncio.create_task(fetch_url(url, sem)) for url in urls]
    results = await asyncio.gather(*tasks)
    print(f"Проверено сайтов: {len(results)}")

asyncio.run(main())

[Запрос] Проверка https://site_1.com началась
[Запрос] Проверка https://site_2.com началась
[Запрос] Проверка https://site_3.com началась
[Готово] https://site_1.com проверен
[Запрос] Проверка https://site_4.com началась
[Готово] https://site_3.com проверен
[Запрос] Проверка https://site_5.com началась
[Готово] https://site_2.com проверен
[Готово] https://site_4.com проверен
[Запрос] Проверка https://site_6.com началась
[Запрос] Проверка https://site_7.com началась
[Готово] https://site_5.com проверен
[Готово] https://site_6.com проверен
[Запрос] Проверка https://site_8.com началась
[Запрос] Проверка https://site_9.com началась
[Готово] https://site_7.com проверен
[Готово] https://site_8.com проверен
[Запрос] Проверка https://site_10.com началась
[Запрос] Проверка https://site_11.com началась
[Готово] https://site_9.com проверен
[Запрос] Проверка https://site_12.com началась
[Готово] https://site_12.com проверен
[Готово] https://site_11.com проверен
[Запрос] Проверка https://site_13.co